# Nexto Distillation — DAgger Round 2

Collects on-policy data with the DAgger-v1 student, retrains with:
- 70% original teacher-driven data
- 30% DAgger-v2 on-policy data
- Warm-started from the DAgger-v1 checkpoint

**Runtime:** Go to **Runtime → Change runtime type → T4 GPU** before starting.

## 1. Setup

In [ ]:
!git clone https://github.com/AdamTheGans/Rocket-League-Bot.git
%cd Rocket-League-Bot

!pip install -q rlgym[rl-sim]
!pip install -q git+https://github.com/AechPro/rlgym-ppo
!pip install -q numpy==1.26.4 cloudpickle==3.1.2

import torch, os
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
assert os.path.isfile('nexto/nexto-model.pt')
print('Setup OK')

## 2. Upload DAgger-v1 Checkpoint

Upload the `student_policy.pt` and `metadata.json` from `checkpoints/nexto_distill_dagger/`.

In [ ]:
import os
os.makedirs('checkpoints/nexto_distill_dagger', exist_ok=True)

from google.colab import files
print('Upload student_policy.pt:')
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'checkpoints/nexto_distill_dagger/{name}', 'wb') as f:
        f.write(data)

print('\nUpload metadata.json:')
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'checkpoints/nexto_distill_dagger/{name}', 'wb') as f:
        f.write(data)

!ls -la checkpoints/nexto_distill_dagger/

## 3. Generate Original Teacher Data (for the 70%)

Regenerate 1M steps of teacher-driven data.

In [ ]:
%cd src

!python -m nexto_distill.generate_dataset \
    --out_dir ../data/nexto_distill/shards \
    --num_steps 1000000 \
    --shard_size 50000 \
    --seed 42 \
    --device cpu \
    --report_every 50000

## 4. DAgger Round 2 Collection

DAgger-v1 student drives, teacher labels. Collect ~430K steps
to get close to a 70/30 ratio with the 1M original steps.

In [ ]:
!python -m nexto_distill.dagger_collect \
    --checkpoint ../checkpoints/nexto_distill_dagger/student_policy.pt \
    --out_dir ../data/nexto_distill/dagger2_shards \
    --num_steps 430000 \
    --shard_size 50000 \
    --seed 123 \
    --report_every 10000

## 5. Retrain: 70% Teacher + 30% DAgger, Warm-Started

- Primary data: original teacher shards (1M, ~900K after val split)
- Extra data: DAgger-v2 shards (430K, all used for training)
- Ratio: ~70/30
- Resume from: DAgger-v1 checkpoint (warm-start)

In [ ]:
!python -m nexto_distill.train_bc \
    --data_dir ../data/nexto_distill/shards \
    --extra_data_dirs ../data/nexto_distill/dagger2_shards \
    --resume_from ../checkpoints/nexto_distill_dagger/student_policy.pt \
    --layers 2048 1024 1024 512 \
    --lr 1e-4 \
    --epochs 20 \
    --batch_size 4096 \
    --checkpoint_dir ../checkpoints/nexto_distill_dagger2 \
    --device cuda \
    --seed 42

## 6. Evaluate DAgger-v2 Student

In [ ]:
# Offline eval
!python -m nexto_distill.eval_imitation \
    --checkpoint ../checkpoints/nexto_distill_dagger2/student_policy.pt \
    --mode offline \
    --data_dir ../data/nexto_distill/shards \
    --device cuda

print('\n' + '='*60 + '\n')

# Online eval
!python -m nexto_distill.eval_imitation \
    --checkpoint ../checkpoints/nexto_distill_dagger2/student_policy.pt \
    --mode online \
    --episodes 50 \
    --device cpu

## 7. Download Results

In [ ]:
!zip -r /content/nexto_distill_dagger2_results.zip \
    ../checkpoints/nexto_distill_dagger2/

from google.colab import files
files.download('/content/nexto_distill_dagger2_results.zip')